# End-to-End Machine Learning Pipeline using Scikit-learn Pipeline API

## Objective

The objective of this notebook is to build a reusable and production-ready machine learning pipeline for predicting customer churn using the Telco Customer Churn dataset.

### Tasks Completed

- Data preprocessing
- Feature scaling
- One-Hot Encoding
- Pipeline construction
- Logistic Regression
- Random Forest
- Hyperparameter tuning using GridSearchCV
- Model evaluation
- Pipeline export using Joblib

In [1]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## Load Dataset

The Telco Customer Churn dataset is loaded into a pandas DataFrame.

In [3]:
df = pd.read_csv("Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Explore Dataset

Inspect dataset shape, data types, missing values, and target distribution.

In [4]:
print(df.shape)

df.info()

df.isnull().sum()

df["Churn"].value_counts()

(7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null

Churn
No     5174
Yes    1869
Name: count, dtype: int64

## Data Cleaning

- Remove customerID
- Convert TotalCharges to numeric
- Handle missing values
- Convert target labels into binary values

In [5]:
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

## Split Features and Target

In [6]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

## Identify Numerical and Categorical Features

Separate numerical and categorical columns so different preprocessing techniques can be applied.

In [7]:
numerical_features = X.select_dtypes(include=["int64","float64"]).columns

categorical_features = X.select_dtypes(include=["object"]).columns

print(numerical_features)

print(categorical_features)

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


## Build Preprocessing Pipeline

The preprocessing pipeline includes:

- Median imputation for numerical features
- Standard Scaling
- Most frequent imputation for categorical features
- One-Hot Encoding

In [8]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

## Split Dataset

Split the dataset into training and testing sets.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Logistic Regression Pipeline

In [10]:
log_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

## Hyperparameter Tuning using GridSearchCV

In [11]:
log_param_grid = {
    "classifier__C":[0.01,0.1,1,10],
    "classifier__penalty":["l2"],
    "classifier__solver":["lbfgs"]
}

log_grid = GridSearchCV(
    log_pipeline,
    log_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

log_grid.fit(X_train,y_train)

print(log_grid.best_params_)

{'classifier__C': 10, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}


## Evaluate Logistic Regression

In [12]:
log_predictions = log_grid.predict(X_test)

print("Accuracy:",accuracy_score(y_test,log_predictions))
print("Precision:",precision_score(y_test,log_predictions))
print("Recall:",recall_score(y_test,log_predictions))
print("F1:",f1_score(y_test,log_predictions))

print(classification_report(y_test,log_predictions))

Accuracy: 0.8055358410220014
Precision: 0.6572327044025157
Recall: 0.5588235294117647
F1: 0.6040462427745664
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



## Random Forest Pipeline

In [13]:
rf_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",RandomForestClassifier(random_state=42))
])

## Hyperparameter Tuning using GridSearchCV

In [14]:
rf_param_grid = {
    "classifier__n_estimators":[100,200],
    "classifier__max_depth":[None,10,20],
    "classifier__min_samples_split":[2,5]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train,y_train)

print(rf_grid.best_params_)

{'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}


## Evaluate Random Forest

In [15]:
rf_predictions = rf_grid.predict(X_test)

print("Accuracy:",accuracy_score(y_test,rf_predictions))
print("Precision:",precision_score(y_test,rf_predictions))
print("Recall:",recall_score(y_test,rf_predictions))
print("F1:",f1_score(y_test,rf_predictions))

print(classification_report(y_test,rf_predictions))

Accuracy: 0.7998580553584103
Precision: 0.6564625850340136
Recall: 0.516042780748663
F1: 0.5778443113772455
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



## Compare Models

In [16]:
results = pd.DataFrame({
    "Model":["Logistic Regression","Random Forest"],
    "Accuracy":[
        accuracy_score(y_test,log_predictions),
        accuracy_score(y_test,rf_predictions)
    ],
    "F1 Score":[
        f1_score(y_test,log_predictions),
        f1_score(y_test,rf_predictions)
    ]
})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.805536,0.604046
1,Random Forest,0.799858,0.577844


## Save the Best Model

Export the best-performing pipeline using Joblib so it can be reused for deployment without rebuilding the preprocessing steps.

In [17]:
best_model = rf_grid if rf_grid.best_score_ > log_grid.best_score_ else log_grid

joblib.dump(best_model, "customer_churn_pipeline.pkl")

print("Pipeline saved successfully.")

Pipeline saved successfully.


## Load Saved Pipeline

In [18]:
loaded_pipeline = joblib.load("customer_churn_pipeline.pkl")

## Make Predictions on New Data

The saved pipeline automatically performs preprocessing before making predictions, demonstrating production-ready deployment.

In [19]:
sample = X_test.iloc[:5]

predictions = loaded_pipeline.predict(sample)

print(predictions)

[0 1 0 0 0]


# Conclusion

In this notebook, an end-to-end machine learning workflow was developed using Scikit-learn's Pipeline API.

Key achievements:

- Automated preprocessing using Pipeline and ColumnTransformer.
- Trained Logistic Regression and Random Forest classifiers.
- Optimized model performance using GridSearchCV.
- Compared multiple models using evaluation metrics.
- Exported the complete preprocessing and modeling pipeline using Joblib.
- Demonstrated how the saved pipeline can be reused for future predictions without additional preprocessing.

This approach follows production-ready machine learning practices by ensuring consistency, reusability, and ease of deployment.